# Tuning Approach — Hybrid ALNS

**What:** Hyperparameter tuning for the Hybrid ALNS solver. We optimise 9 metaheuristic knobs over a bank of 100 bin-packing instances to minimise solution gap at a reasonable compute cost.

---

## 1. Objective

The Optuna optimiser minimises a **composite objective**:

```
composite = 0.9 × mean_relative_gap + 0.1 × mean_normalized_time
```

where

```
mean_relative_gap    = avg((bins_used − lb) / max(lb, 1))      (gap, scale-invariant)
mean_normalized_time = avg(elapsed / time_limit)                 (time, relative to budget)
```

- **90% weight on gap** — solution quality is primary
- **10% weight on time** — discourages unnecessarily slow configurations
- Relative gap denominator `max(lb, 1)` guards against division by zero
- Both metrics are averaged over the 100-instance bank

---

## 2. Parameter Taxonomy

The solver has 15 parameters. We split them into two groups.

### 9 Tunable Parameters

| Parameter | Default | Stage-1 Range | Role |
|---|---|---|---|
| `initial_temperature` | 1.44 | (0.5, 5.0) log | SA starting temp — controls early acceptance rate |
| `alpha_cool` | 0.9995 | (0.998, 0.9999) | Cooling factor — slower = more exploration |
| `k_min_frac` | 0.05 | (0.01, 0.20) | Min destruction radius (fraction of items) |
| `k_max_frac` | 0.25 | (0.10, 0.40) | Max destruction radius; grows with stagnation |
| `bandit_alpha` | 0.30 | (0.05, 1.0) log | LinUCB exploration bonus |
| `warmup_calls` | 300 | (50, 600) int | Thompson Sampling warm-up before LinUCB |
| `no_improve_frac` | 0.05 | (0.01, 0.20) | Stagnation budget (fraction of max iter) |
| `reheat_soft_mult` | 0.35 | (0.10, 0.90) | Temp multiplier for periodic soft reheats |
| `reheat_hard_mult` | 0.20 | (0.05, 0.80) | Temp multiplier for hard restarts |

These are the **metaheuristic control surface** — they determine how the solver explores vs exploits.

### 6 Structural Parameters (Held Fixed)

| Parameter | Default | Why Fixed |
|---|---|---|
| `min_no_improve_limit` | 250 | Safety floor; real lever is `no_improve_frac` |
| `temp_precision_floor` | 1e-12 | Division guard for T/T0 feature |
| `reheat_check_interval_divisor` | 4.0 | Tied to `no_improve_frac` |
| `reheat_check_min_interval` | 50 | Prevents thrashing on short runs |
| `hard_restart_min_limit` | 100 | Prevents restart thrashing |
| `patience_shrink_factor` | 2/3 | Shrink factor on hard restart (changed from 0.5) |

These are safety / design constants. Leaving them at defaults avoids secondary interactions that would confuse the optimiser.

---

## 3. Pipeline: 3-Stage Tuning

We never throw all 9 knobs into Optuna at once. Instead we use a three-stage pipeline:

### Stage 0 — Isolation Grid Search

Before Bayesian optimisation, we run three **independent grid searches**, one per parameter group. The bandit is forced to **uniform random** (`force_uniform_random=True`) so the bandit's adaptivity does not confound the measurement.

| Group | Parameters | Grid | Constraint |
|---|---|---|---|
| SA Thermal | `initial_temperature`, `alpha_cool`, `reheat_soft_mult`, `reheat_hard_mult` | 3×3×3×3 = 81 → 54 valid | `reheat_hard_mult < reheat_soft_mult` |
| Destruction | `k_min_frac`, `k_max_frac`, `no_improve_frac` | 3×3×3 = 27 | `k_min_frac < k_max_frac` |
| Bandit | `bandit_alpha`, `warmup_calls` | 3×3 = 9 | none |

Each grid is evaluated on a **20-instance isolation bank** at 2 000 ALNS iterations. The best config from each group is merged into a single "after isolation" config.

**Why uniform-random bandit for steps 1 & 2?**  
If the bandit were active, it would learn preferences during the grid search. Those preferences depend on which grid point is currently being evaluated, creating a feedback loop. Forcing uniform random removes this confound and gives an unbiased picture of each group's standalone effect.

**How the isolation result feeds into Stage 1:**

The merged isolation best is **enqueued** into Stage 1's Optuna study via `study.enqueue_trial()`. Enqueuing means: "evaluate this exact parameter configuration as one of the 50 trials before the TPE sampler generates any candidates." It is not a hint or a manual override — it is a real, evaluated trial that TPE's probabilistic surrogate model learns from.

- **Trial 0** = the default configuration (always enqueued, giving TPE a reference point)
- **Trial 1** = the isolation-merged best (enqueued if isolation was run)
- **Trials 2–49** = sampled by TPE based on what it learned from trials 0 and 1

**Crucially, the isolation result does not shrink Stage 1's search ranges.** Stage 1 still searches the full `STAGE1_RANGES` (see the parameter table in section 2). What the enqueued trial does is give TPE's model an extra known-good data point, so the model learns "this region is promising" and allocates more samples nearby. The range boundaries are unchanged.

This is different from Stage 2, where the ranges **are** physically narrowed around Stage 1's best (see below).

### Stage 1 — Coarse Bayesian Search (Optuna)

Stage 1 explores the full parameter space broadly, identifying the most promising basin.

- **50 trials** (trial 0 = default, trial 1 = isolation best if available, rest sampled by TPE)
- **5 000 ALNS iterations** per instance
- **100-instance bank**
- **TPE sampler** with 10 startup trials
- **Wide search ranges** (unchanged from `STAGE1_RANGES` — see the table in section 2)
- **Pruning:** trials are pruned if `k_min >= k_max` or `reheat_hard >= reheat_soft`
- **Composite objective** with 90/10 gap/time weighting

The enqueued isolation best (trial 1) serves as a training point for TPE's surrogate model. It influences **where** TPE samples more densely, but not **how far** it can search. The boundary of every parameter remains the full Stage-1 range.

### Stage 2 — Fine Bayesian Search (Optuna)

Stage 2 refines within the best region identified by Stage 1. Unlike the isolation-to-Stage-1 handoff, Stage 2 **physically narrows the search ranges**.

**How the narrowing works:**

Optuna receives a computed set of narrow ranges (`STAGE2_RANGES`) instead of the wide `STAGE1_RANGES`. For each parameter, Stage 1's best value becomes the centre of a window:

- Parameters with wide, scale-free ranges use **multiplicative offsets**: ±15–25% of the anchor value (e.g. `initial_temperature` best = 1.8 → range [1.35, 2.25])
- Parameters with narrow or integer ranges use **additive offsets**: ±0.02 or ±80 units (e.g. `k_min_frac` best = 0.04 → range [0.02, 0.06])
- Upper bounds are clamped by solver-imposed limits where necessary (e.g. `alpha_cool < 1.0`)

The anchor config (Stage 1's best parameters, merged with defaults) is **enqueued as trial 0**, exactly like the isolation best was enqueued in Stage 1. But now the search is confined to the narrow window — TPE cannot propose values outside it.

- **50 trials** (trial 0 = Stage 1 best enqueued, trials 1–49 sampled by TPE)
- **5 000 ALNS iterations** per instance
- **100-instance bank**
- **Narrow ranges** computed as ±15–25% around Stage 1's best
- **Same pruner and objective** as Stage 1
- The **default config is NOT re-enqueued** (Stage 2 trusts the basin found in Stage 1)

### Final Evaluation

After Stage 2, the best configuration found is **re-evaluated on the full 100-instance bank** with the same budget. This gives a clean comparison against the baseline (which was also evaluated on 100 instances at 2 000 iterations).

---

## 4. Instance Bank

From the 710 available instances across 5 datasets (`falkenauer-t`, `falkenauer-u`, `scholl-1`, `scholl-2`, `scholl-3`), we sample **100 instances stratified by item-count buckets**:

- Small: < 100 items
- Medium: 100–500 items  
- Large: > 500 items

Stratification ensures coverage across problem sizes without overfitting to any single dataset. The same 100-instance bank is used for every evaluation in the pipeline.

---

## 5. Why This Design?

| Choice | Rationale |
|---|---|
| **Isolation first** | Reduces dimensionality for Optuna; gives unbiased per-group signal |
| **Uniform bandit during isolation** | Prevents bandit adaptivity from confounding grid results |
| **Two Optuna stages** | Wide exploration (S1) → narrow refinement (S2) is a standard Bayesian optimisation best practice |
| **Enqueue isolation best** (S1) | Gives TPE a known-good data point without narrowing the search space — influences sampling density, not range boundaries |
| **Enqueue Stage-1 best** (S2) | Anchors TPE at the best known basin; ranges are physically narrowed so TPE cannot drift back to wide exploration |
| **Composite objective** | Pure gap minimisation can produce impractically slow configs; the 10% time weight keeps them fast enough |
| **Structural params fixed** | Fewer dimensions = faster convergence; these values are safe at defaults |
| **Pruning constraints** | Avoids spending iterations on invalid regions (`k_min ≥ k_max`, `h ≥ s`) |
| **Instance bank stratification** | Ensures results generalise across problem sizes |

---

## 6. Output File Flow

```
tune_with_optuna.py
│
├── baseline_defaults.json            ← Default config evaluated on 100 instances (2000 iter)
│
├── isolation_step1_sa_thermal.json   ← Best SA params from grid (20 instances, 2000 iter)
├── isolation_step2_destruction.json  ← Best destruction params from grid
├── isolation_step3_bandit.json       ← Best bandit params from grid
├── isolation_merged.json             ← All three merged + re-evaluated (20 instances)
│
├── alns_stage1_coarse_study.json     ← Full Optuna study: 50 trials, wide ranges, 5000 iter
├── alns_stage2_fine_study.json       ← Full Optuna study: 50 trials, narrow windows, 5000 iter
│
├── best_config_evaluation.json       ← Best config re-evaluated on 100 instances (5000 iter)
└── pipeline_summary.json             ← CLI args, wall timing, improvement %
```

The results notebook (`parameter_tuning.ipynb`) loads these JSON files and visualises the outcome at every stage.

